In [1]:
import torch 
import torch.nn as nn 
import torch.nn.functional as F 
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader
from torch.optim import Adam
import torchvision
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

torch.manual_seed(42)

# Baseline:
# zScore Normalization
# 4 blocks (2, 2, 3, 3)
# ReLU, He Initialization
# Adam, lr = 1e-3

In [2]:
batch_size = 256
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.4914, 0.4822, 0.4465],
                         [0.2470, 0.2435, 0.2616])
])
train_set = CIFAR10(
    root = 'data',
    train = True,
    download = True,
    transform = transform
)
test_set = CIFAR10(
    root = 'data',
    train = False, 
    download = True,
    transform = transform
)
trainloader = DataLoader(
    train_set,
    batch_size = batch_size,
    shuffle = True,
    num_workers = 2
)
testloader = DataLoader(
    test_set,
    batch_size = batch_size,
    shuffle = True,
    num_workers = 2
)


In [3]:
class CNNModel(nn.Module):
    def __init__(self, n_classes):
        super(CNNModel, self).__init__()
        self.conv_layer1 = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride = 1, padding = 'same'),
            nn.ReLU()
        )
        self.conv_layer2 = nn.Sequential(
            nn.Conv2d(32, 32, 3, stride = 1, padding = 'same'),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.conv_layer3 = nn.Sequential(
            nn.Conv2d(32, 64, 3, stride = 1, padding = 'same'),
            nn.ReLU()
        )
        self.conv_layer4 = nn.Sequential(
            nn.Conv2d(64, 64, 3, stride = 1, padding = 'same'),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.conv_layer5 = nn.Sequential(
            nn.Conv2d(64, 128, 3,stride = 1, padding = 'same'),
            nn.ReLU()
        )
        self.conv_layer6 = nn.Sequential(
            nn.Conv2d(128, 128, 3, stride = 1, padding = 1),
            nn.ReLU()
        )
        self.conv_layer7 = nn.Sequential(
            nn.Conv2d(128, 128, 3, stride = 1, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.conv_layer8 = nn.Sequential(
            nn.Conv2d(128, 256, 3, stride = 1, padding = 1),
            nn.ReLU()
        )
        self.conv_layer9 = nn.Sequential(
            nn.Conv2d(256, 256, 3, padding = 'same', stride = 1),
            nn.ReLU()
        )
        self.conv_layer10 = nn.Sequential(
            nn.Conv2d(256, 256, 3, padding = 'same',stride = 1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 2 * 2, 512),
            nn.ReLU(),
            nn.Linear(512, n_classes)
        )
    def forward(self, x):
        x = self.conv_layer1(x)
        x = self.conv_layer2(x)
        x = self.conv_layer3(x)
        x = self.conv_layer4(x)
        x = self.conv_layer5(x)
        x = self.conv_layer6(x)
        x = self.conv_layer7(x)
        x = self.conv_layer8(x)
        x = self.conv_layer9(x)
        x = self.conv_layer10(x)
        out = self.classifier(x)
        return out     

In [4]:
model = CNNModel(10)
model.to(device)
print(model)

CNNModel(
  (conv_layer1): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (1): ReLU()
  )
  (conv_layer2): Sequential(
    (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_layer3): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (1): ReLU()
  )
  (conv_layer4): Sequential(
    (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_layer5): Sequential(
    (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (1): ReLU()
  )
  (conv_layer6): Sequential(
    (0): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
  )
  (conv_layer7): Sequential(
    (0): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 

In [5]:
criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr = 1e-3)

In [6]:
train_losses = []
train_accuracies = []
test_losses = []
test_accuracies = []

In [7]:
def evaluate(model, testloader, criterion):
    model.eval()
    test_loss = 0.0
    running_correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            test_loss += loss.item()

            predicted = torch.argmax(outputs.data, dim = 1)
            total += labels.size(0)
            running_correct += (predicted == labels).sum().item()
    accuracy = 100 * running_correct / total
    total_loss = test_loss / len(testloader)
    return total_loss, accuracy

In [ ]:
epochs = 100
for epoch in range(epochs):
    model.train()
    train_loss =  0.0
    correct = 0
    total = 0
    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        train_loss += loss.item() * labels.size(0)
        loss.backward()

        optimizer.step()

        predicted = torch.argmax(outputs.data, dim = 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_accuracy = 100 * correct / total
    epoch_loss = train_loss / total
    test_loss, test_accuracy = evaluate(model, testloader, criterion)
    print(f'Epoch: {epoch + 1}/{epochs}, Loss: {epoch_loss:.4f}, Accuracy : {epoch_accuracy :.2f}%, Test loss: {test_loss}, Test accuracy: {test_accuracy}%')
    train_losses.append(epoch_loss)
    train_accuracies.append(epoch_accuracy)
    test_losses.append(test_loss)
    test_accuracies.append(test_accuracy)


Epoch: 1/100, Loss: 100.0000, Accuracy : 25.17%, Test loss: 1.6282579272985458, Test accuracy: 37.7%
Epoch: 2/100, Loss: 100.0000, Accuracy : 44.75%, Test loss: 1.384196600317955, Test accuracy: 48.13%
Epoch: 3/100, Loss: 100.0000, Accuracy : 56.08%, Test loss: 1.097769594192505, Test accuracy: 60.26%
Epoch: 4/100, Loss: 100.0000, Accuracy : 63.76%, Test loss: 1.0675575777888298, Test accuracy: 62.55%
Epoch: 5/100, Loss: 100.0000, Accuracy : 68.50%, Test loss: 0.8778121173381805, Test accuracy: 68.7%


In [ ]:
plt.plot(train_losses, labels = 'Train Losses')
plt.plot(test_losses, labels = ' Test Losses')
plt.legend()

In [ ]:
plt.plot(train_accuracies, labels = 'Train Accuracies')
plt.plot(test_accuracies, labels = 'Test Accuracies')
plt.legend()